# WildFL — Federated Learning for Wildlife Detection

This notebook demonstrates how to run the federated-learning (FL) algorithms implemented in `algorithms/`:
**FedAvg**, **FedBN**, **FedNova**, **MOON**, **FedDC**, and **FedProto**.

All algorithms collaboratively train a YOLO bounding-box detector across several *clients* (e.g., wildlife parks or
survey campaigns), each holding its own local set of annotated aerial images. Raw images never leave a client:
in every communication round, clients train locally and share only model parameters (or, in the case of FedProto,
class prototypes), which the server then aggregates.

**Contents**

1. Prerequisites
2. Expected data layout
3. Shared configuration
4. Running the algorithms
5. Inspecting the results
6. References

## 1. Prerequisites

The algorithms federate [Ultralytics](https://docs.ultralytics.com/) YOLO detectors. Two customized packages are
required in addition to the usual scientific Python stack:

- **Customized `ultralytics` fork.** In every round, local training must start from the aggregated server weights.
  The algorithms therefore pass an additional `state=<path to a state dict>` argument to `model.train()`, which
  stock Ultralytics does not support. MOON and FedProto further rely on the fork's model-contrastive and
  prototype-alignment loss terms.
- **Customized `sahi` fork.** Models are evaluated on full-size aerial images via
  [SAHI](https://github.com/obss/sahi) tiled inference; the fork accepts the additional inference arguments
  forwarded by the algorithms.

The helper module (`helpers.py`) and the algorithms additionally require `torch`, `opencv-python`, `numpy`,
`pandas`, `matplotlib`, `tqdm`, and `openpyxl`. A CUDA-capable GPU is required — training and tiled inference
assume device `cuda:0`.

> **Note.** The cells in Section 4 launch complete federated trainings, which take hours on a single GPU. They are
> meant as templates: point `data_dir` and `cfg_dir` in the shared configuration (Section 3) at your own setup
> before executing them.

In [ ]:
import sys
import json

from pathlib import Path

# make the algorithm classes importable (the notebook runs from the repository root)
sys.path.append("algorithms")

from helpers import init_rng, NAMES_DIRECTORIES_FILES
from FedAvg import FedAvg
from FedBN import FedBN
from FedNova import FedNova
from MOON import MOON
from FedDC import FedDC
from FedProto import FedProto

init_rng(0)  # seed the Python, NumPy and PyTorch RNGs for reproducibility

## 2. Expected data layout

All algorithms operate on a prepared dataset directory (`data_dir`) containing one subdirectory per client.
Each client holds image *patches* for training (in Ultralytics YOLO format) and *full-size images* for
evaluation:

```text
data_dir/
├── client_stats.json            # per-client dataset statistics (used to weight the aggregation)
├── c1/
│   ├── dataset.yaml             # Ultralytics dataset file, pointing to the train/val patch directories
│   ├── train/                   # training patches (images + YOLO label files)
│   ├── val/                     # validation patches
│   └── val_full_imgs/           # evaluation split: full-size images
│       ├── annotations.json     # image name -> list of box annotations
│       ├── dataset_SAHI.json    # dataset description for tiled (SAHI) inference
│       └── image_counts/        # one .json per image with ground-truth counts per class
├── c2/
│   └── ...
└── all_data/                    # optional "centralized" client: the union of all clients' data
    └── ...
```

Three files deserve special attention:

- **`client_stats.json`** (top level) describes every client's data distribution. The algorithms read the number
  of training *annotations* per class to derive each client's aggregation weight (FedNova additionally uses the
  patch counts to determine the number of local optimization steps):

  ```json
  {
   "c1": {"data": {"train": {"anns": {"0": 4018},
                             "images": 130,
                             "patches": {"patches_kept": 1697, "...": "..."}},
                   "val":   {"...": "..."}}},
   "c2": {"...": "..."}
  }
  ```

- **`annotations.json`** (inside the evaluation split) maps every image name to its list of annotations. With
  `ann_format="BX_WH"`, a single annotation reads
  `{"bbox": [x_min, y_min, width, height], "category_id": 0}`.

- **`dataset.yaml`** is the standard Ultralytics dataset file of each client. The number of classes in the model
  configuration YAML is patched automatically before training (`helpers.modify_cfg_yaml_nc`).

The optional `all_data` client contains the pooled data of all clients; FedNova and MOON evaluate the aggregated
global model on it as a centralized reference.

All canonical file and directory names are defined in `helpers.NAMES_DIRECTORIES_FILES`:

In [ ]:
for role, name in NAMES_DIRECTORIES_FILES.items():
    print(f"{role:<22} ->  {name}")

## 3. Shared configuration

All experiments below share one configuration dictionary. The keys mirror the arguments of the algorithms'
`__call__` methods:

| Key | Meaning |
|---|---|
| `rounds` | number of communication rounds |
| `mdl_type`, `mdl_version` | Ultralytics model family and size; the model is built from `<mdl_type><mdl_version>.yaml` (e.g. `yolo11n.yaml`) |
| `clients` | names of the client subdirectories inside `data_dir` |
| `data_dir` | root of the prepared dataset (Section 2) |
| `cfg_dir` | directory of the Ultralytics model-configuration YAMLs, i.e. the folder containing `<mdl_type>.yaml` |
| `class_id2name` | mapping of integer class ids to class names |
| `ann_format` | annotation format of `annotations.json`; `"BX_WH"` denotes `[x_min, y_min, width, height]` boxes |
| `training_params` | forwarded unchanged to Ultralytics' `model.train()` for every client |
| `img_format` | image file extension |
| `eval_split` | subdirectory of each client used for evaluation |
| `use_tiling` | if `True`, evaluation images are processed patch-wise with SAHI tiled inference |
| `eval_global` | if `True`, the aggregated global model is additionally evaluated on every client after each round |
| `patch_dims`, `patch_overlap` | tile size and overlap used for tiled inference |

> **Why `val` must be `False`:** with validation enabled, Ultralytics restores the *best* checkpoint at the end of
> local training — but federated aggregation must combine the *last* local weights. Every algorithm asserts
> `training_params["val"] == False` for this reason; `patience` is set high to disable early stopping as well.

In [ ]:
config = {
    "rounds":        30,
    "mdl_type":      "yolo11",
    "mdl_version":   "n",
    "clients":       ["c1", "c2", "c3", "c4", "c5"],
    "data_dir":      "/path/to/prepared_dataset",           # <-- adapt: see Section 2 for the expected layout
    "cfg_dir":       "/path/to/ultralytics/cfg/models/11",  # <-- adapt: folder containing yolo11.yaml
    "class_id2name": {0: "Elephant"},
    "ann_format":    "BX_WH",
    "training_params": {
        "epochs":   20,      # local epochs per round
        "batch":    32,
        "imgsz":    640,
        "device":   0,
        "patience": 1000,    # effectively disables early stopping
        "amp":      False,
        "val":      False,   # must be False (see the note above)
    },
    "img_format":    "jpg",
    "eval_split":    "val_full_imgs",
    "use_tiling":    True,
    "eval_global":   True,
    "patch_dims":    {"width": 640, "height": 640},
    "patch_overlap": 0.2,
}


def make_output_dir(experiment_name: str) -> str:
    """Create an experiment directory and store the configuration alongside the results."""
    out = Path("experiments") / experiment_name
    out.mkdir(parents=True, exist_ok=True)
    with open(out / "config.json", "w") as f:
        json.dump(config, f, indent=1)
    return str(out)

## 4. Running the algorithms

All six algorithms share the same two-step interface:

```python
algo = Algorithm(<model / schedule configuration>)   # 1. construct
results = algo(<data + training configuration>)      # 2. run the federation
```

Every communication round proceeds in three steps:

1. **Local training** — each participating client updates a model on its local training patches, starting from the
   current server state.
2. **Aggregation** — the server combines the client updates; the weight of each client is proportional to its
   number of training annotations (read from `client_stats.json`).
3. **Evaluation** — every local model is evaluated on its own client's `eval_split`; with `eval_global=True`, the
   aggregated global model is additionally evaluated on all clients.

The per-round Ultralytics training runs are written to `<output_dir>/round<i>/<client>/` (pass
`rm_client_data=True` to delete them after every round and save disk space). Metrics are streamed to
`<output_dir>/result_<algorithm>.json` (global model) and `<output_dir>/result_<algorithm>_local.json`
(local models) after every round, so intermediate results survive interruptions.

### 4.1 FedAvg

FedAvg (McMahan et al., 2017) is the reference federated-learning algorithm: after each round of local training,
the server replaces the global model with the *weighted average* of the clients' parameters, and the averaged model
is redistributed as the starting point of the next round. In this implementation, client weights are computed from
the number of local training annotations (rather than the number of images), which extends naturally to
multi-class datasets.

In [ ]:
fedavg = FedAvg(rounds=config["rounds"], mdl_type=config["mdl_type"], mdl_version=config["mdl_version"])

output_dir = make_output_dir("FedAvg_20E_30R")

results = fedavg(clients=config["clients"], data_dir=config["data_dir"], cfg_dir=config["cfg_dir"],
                 class_id2name=config["class_id2name"], ann_format=config["ann_format"],
                 training_params=config["training_params"], img_format=config["img_format"],
                 eval_split=config["eval_split"], use_tiling=config["use_tiling"], output_dir=output_dir,
                 patch_dims=config["patch_dims"], patch_overlap=config["patch_overlap"],
                 eval_global=config["eval_global"])

#### Partial participation

FedAvg additionally supports a **client schedule** that lets clients join the federation at different points in
time — useful to study, for example, a new survey site joining an ongoing federation. Each client is assigned a
`join_round` and participates from that round until the end. The total number of rounds is derived from the
schedule (so `rounds` can be omitted in the constructor), and the aggregation weights are re-normalized every round
over the currently active clients.

In [ ]:
client_schedule = {"c1": {"join_round": 0,  "rounds": 30},
                   "c2": {"join_round": 0,  "rounds": 30},
                   "c3": {"join_round": 10, "rounds": 20},
                   "c4": {"join_round": 15, "rounds": 15},
                   "c5": {"join_round": 20, "rounds": 10}}

fedavg = FedAvg(mdl_type=config["mdl_type"], mdl_version=config["mdl_version"])  # rounds derived from the schedule

output_dir = make_output_dir("FedAvg_20E_staggered")

results = fedavg(clients=config["clients"], data_dir=config["data_dir"], cfg_dir=config["cfg_dir"],
                 class_id2name=config["class_id2name"], ann_format=config["ann_format"],
                 training_params=config["training_params"], img_format=config["img_format"],
                 eval_split=config["eval_split"], use_tiling=config["use_tiling"], output_dir=output_dir,
                 patch_dims=config["patch_dims"], patch_overlap=config["patch_overlap"],
                 eval_global=config["eval_global"], client_schedule=client_schedule)

### 4.2 FedBN

FedBN (Li et al., 2021) targets *feature-shift* heterogeneity — clients whose images differ in appearance (sensor,
altitude, season, vegetation) rather than in label distribution, which is typical for aerial wildlife surveys. All
parameters are averaged exactly as in FedAvg, **except the BatchNorm layers, which stay local** to each client.
Every client therefore keeps a personalized model: the shared, aggregated backbone combined with its own
normalization statistics. With `eval_global=True`, these personalized models (shared parameters + local BatchNorm
state) are evaluated on all clients after every round.

In [ ]:
fedbn = FedBN(rounds=config["rounds"], mdl_type=config["mdl_type"], mdl_version=config["mdl_version"])

output_dir = make_output_dir("FedBN_20E_30R")

results = fedbn(clients=config["clients"], data_dir=config["data_dir"], cfg_dir=config["cfg_dir"],
                class_id2name=config["class_id2name"], ann_format=config["ann_format"],
                training_params=config["training_params"], img_format=config["img_format"],
                eval_split=config["eval_split"], use_tiling=config["use_tiling"], output_dir=output_dir,
                patch_dims=config["patch_dims"], patch_overlap=config["patch_overlap"],
                eval_global=config["eval_global"])

### 4.3 FedNova

When clients hold different amounts of data, the same number of local epochs translates into different numbers of
optimization steps — and naive averaging then drifts toward the clients that take more steps (the *objective
inconsistency* problem). FedNova (Wang et al., 2020) corrects for this by normalizing each client's cumulative
update by its number of local steps before averaging. Here, the number of steps is derived from each client's
patch count in `client_stats.json` together with the batch size and local epochs.

In [ ]:
fednova = FedNova(rounds=config["rounds"], mdl_type=config["mdl_type"], mdl_version=config["mdl_version"])

output_dir = make_output_dir("FedNova_20E_30R")

results = fednova(clients=config["clients"], data_dir=config["data_dir"], cfg_dir=config["cfg_dir"],
                  class_id2name=config["class_id2name"], ann_format=config["ann_format"],
                  training_params=config["training_params"], img_format=config["img_format"],
                  eval_split=config["eval_split"], use_tiling=config["use_tiling"], output_dir=output_dir,
                  patch_dims=config["patch_dims"], patch_overlap=config["patch_overlap"],
                  eval_global=config["eval_global"])

### 4.4 MOON

MOON — *model-contrastive federated learning* (Li, He & Song, 2021) — adds a contrastive term to the local training
objective: the representation of the current local model is pulled toward that of the global model and pushed away
from that of the previous local model, which limits client drift on heterogeneous data. Two extra hyperparameters
are added to `training_params`: `moon` (µ, the weight of the contrastive term) and `tau_moon` (the temperature of
the contrastive loss).

> **Implementation note.** Before each round, MOON pre-computes the feature embeddings that the contrastive loss
> compares against (those of the global and of the previous local model) for every training patch, and stores them
> as `.npy` files inside the client training directories. Use a dedicated copy of `data_dir` if the original should
> stay untouched.

In [ ]:
moon_training_params = dict(config["training_params"])
moon_training_params.update({"moon": 1.0,        # weight of the model-contrastive loss term
                             "tau_moon": 0.5})   # temperature of the contrastive loss

moon = MOON(rounds=config["rounds"], mdl_type=config["mdl_type"], mdl_version=config["mdl_version"])

output_dir = make_output_dir("MOON_1.0MU_0.5TAU_20E_30R")

results = moon(clients=config["clients"], data_dir=config["data_dir"], cfg_dir=config["cfg_dir"],
               class_id2name=config["class_id2name"], ann_format=config["ann_format"],
               training_params=moon_training_params, img_format=config["img_format"],
               eval_split=config["eval_split"], use_tiling=config["use_tiling"], output_dir=output_dir,
               patch_dims=config["patch_dims"], patch_overlap=config["patch_overlap"],
               eval_global=config["eval_global"])

### 4.5 FedDC

FedDC — *federated daisy-chaining* (Kamp, Fischer & Vreeken, 2023) — was designed for federations with **small local
datasets**. Between aggregation events, models are not only averaged but also *passed around from client to
client* in random order, so every model effectively sees more data than any single client holds. The constructor
takes a different set of arguments:

- `agg_events` — number of aggregation events (one "round" in the result files corresponds to one aggregation);
- `dc_period` — number of local epochs a model trains on one client before being passed to the next;
- `agg_period` — number of epochs between two aggregations (must be a multiple of `dc_period`).

Because the training length is fully determined by the two periods, `epochs` **must not** appear in
`training_params` for this algorithm.

In [ ]:
feddc_training_params = {k: v for k, v in config["training_params"].items() if k != "epochs"}

feddc = FedDC(agg_events=30, dc_period=4, agg_period=20,
              mdl_type=config["mdl_type"], mdl_version=config["mdl_version"])

output_dir = make_output_dir("FedDC_4DC_20AGG_30R")

results = feddc(clients=config["clients"], data_dir=config["data_dir"], cfg_dir=config["cfg_dir"],
                class_id2name=config["class_id2name"], ann_format=config["ann_format"],
                training_params=feddc_training_params, img_format=config["img_format"],
                eval_split=config["eval_split"], use_tiling=config["use_tiling"], output_dir=output_dir,
                patch_dims=config["patch_dims"], patch_overlap=config["patch_overlap"],
                eval_global=config["eval_global"])

### 4.6 FedProto

FedProto (Tan et al., 2022) exchanges **class prototypes instead of model weights**: after local training, each
client computes the mean feature embedding of its annotated objects per class, and the server aggregates these into
global prototypes. From the second round on, an alignment term regularizes local training by pulling each client's
object embeddings toward the global prototypes. Model parameters never leave the clients — every client continues
training its own personalized model from its previous state — which drastically reduces communication and preserves
personalization.

The call signature differs slightly from the other algorithms:

- `measure_performance` replaces `eval_global` — there is no global model, so only the personalized client models
  are evaluated;
- `min_iou_eval` sets the IoU threshold at which a detection counts as a true positive during evaluation;
- `inf_params` forwards optional inference arguments to the tiled evaluation (an empty dict uses the defaults).

In [ ]:
fedproto = FedProto(rounds=config["rounds"], mdl_type=config["mdl_type"], mdl_version=config["mdl_version"])

output_dir = make_output_dir("FedProto_20E_30R")

results = fedproto(clients=config["clients"], data_dir=config["data_dir"], cfg_dir=config["cfg_dir"],
                   class_id2name=config["class_id2name"], ann_format=config["ann_format"],
                   training_params=config["training_params"], inf_params={},
                   img_format=config["img_format"], eval_split=config["eval_split"],
                   use_tiling=config["use_tiling"], output_dir=output_dir,
                   patch_dims=config["patch_dims"], patch_overlap=config["patch_overlap"],
                   measure_performance=True, min_iou_eval=0.5)

## 5. Inspecting the results

All algorithms return — and continuously write — nested dictionaries of per-round metrics, evaluated per client:

```json
{
 "round0": {
  "c1": {
   "count_err": {"0": {"MAE": "...", "ME": "...", "MSE": "...", "MARE": "...", "MRE": "..."},
                 "binary": {"...": "..."}},
   "em":        {"0": {"confusion": "...", "precision": "...", "recall": "...", "f1": "...",
                       "metrics/precision(B)": "...", "metrics/recall(B)": "...",
                       "metrics/mAP50(B)": "...", "metrics/mAP50-95(B)": "..."},
                 "binary": {"...": "..."}}
  },
  "c2": "..."
 },
 "round1": "..."
}
```

Metrics are reported per class id (`"0"`, `"1"`, ...) and class-agnostic (`"binary"`), in two groups:

| Group | Metrics | Meaning |
|---|---|---|
| `count_err` | `MAE`, `ME`, `MSE`, `MARE`, `MRE` | image-level counting errors: predicted vs. ground-truth animal counts per image (mean absolute / signed / squared / relative errors) |
| `em` | `precision`, `recall`, `f1`, `confusion` | detection metrics aggregated from image-level confusion matrices (`confusion` is the per-class confusion rate, in %) |
| `em` | `metrics/precision(B)`, `metrics/recall(B)`, `metrics/mAP50(B)`, `metrics/mAP50-95(B)` | standard Ultralytics box-detection metrics |

The helper `get_single_metric_single_client` extracts the trajectory of one metric for one client across all
rounds, which makes plotting straightforward:

In [ ]:
import matplotlib.pyplot as plt

from helpers import get_single_metric_single_client

RESULT_FILE = "experiments/FedAvg_20E_30R/result_fedavg.json"   # written by the FedAvg run above
METRIC = "metrics/mAP50(B)"                                     # any metric from the tables above
CATEGORY = "0"                                                  # class id as a string, or "binary"

with open(RESULT_FILE, "r") as f:
    results = json.load(f)

# fixed, colorblind-safe client colors
client_colors = ["#2a78d6", "#008300", "#e87ba4", "#eda100", "#1baf7a"]

fig, ax = plt.subplots(figsize=(9, 5))

for client, color in zip(config["clients"], client_colors):
    trajectory = get_single_metric_single_client(results=results, metric=METRIC, cats=[CATEGORY], client=client)
    ax.plot(range(len(trajectory[CATEGORY]["global"])), trajectory[CATEGORY]["global"],
            color=color, linewidth=2, label=client)

ax.set_xlabel("Communication round")
ax.set_ylabel(METRIC)
ax.legend(title="Client", frameon=False)
ax.spines[["top", "right"]].set_visible(False)
ax.grid(axis="y", color="#e1e0d9", linewidth=0.8)
ax.set_axisbelow(True)
fig.tight_layout()
plt.show()

A compact tabular summary of the final round complements the learning curves:

In [ ]:
import pandas as pd

final_round = f"round{len(results) - 1}"
metrics_to_tabulate = ["MAE", "precision", "recall", "f1", "metrics/mAP50(B)"]

rows = {}
for client in config["clients"]:
    client_res = results[final_round][client]
    rows[client] = {m: (client_res["count_err"][CATEGORY][m] if m in client_res["count_err"][CATEGORY]
                        else client_res["em"][CATEGORY][m])
                    for m in metrics_to_tabulate}

pd.DataFrame(rows).T.round(3)

## 6. References

- McMahan, B., Moore, E., Ramage, D., Hampson, S., & Agüera y Arcas, B. (2017).
  *Communication-Efficient Learning of Deep Networks from Decentralized Data.* AISTATS 2017.
- Li, X., Jiang, M., Zhang, X., Kamp, M., & Dou, Q. (2021).
  *FedBN: Federated Learning on Non-IID Features via Local Batch Normalization.* ICLR 2021.
- Wang, J., Liu, Q., Liang, H., Joshi, G., & Poor, H. V. (2020).
  *Tackling the Objective Inconsistency Problem in Heterogeneous Federated Optimization.* NeurIPS 2020.
- Li, Q., He, B., & Song, D. (2021).
  *Model-Contrastive Federated Learning.* CVPR 2021.
- Kamp, M., Fischer, J., & Vreeken, J. (2023).
  *Federated Learning from Small Datasets.* ICLR 2023.
- Tan, Y., Long, G., Liu, L., Zhou, T., Lu, Q., Jiang, J., & Zhang, C. (2022).
  *FedProto: Federated Prototype Learning across Heterogeneous Clients.* AAAI 2022.